In [1]:
from datasets import Dataset
from pathlib import Path

output_dir = Path('data/cuad/reformatted')
test_dataset = Dataset.from_json(str(output_dir / 'test.json'))
test_dataset[0]

{'context': 'Exhibit 10.16 SUPPLY CONTRACT Contract No: Date: The buyer/End-User: Shenzhen LOHAS Supply Chain Management Co., Ltd. ADD: Tel No. : Fax No. : The seller: ADD: The Contract is concluded and signed by the Buyer and Seller on , in Hong Kong. 1. General provisions 1.1 This is a framework agreement, the terms and conditions are applied to all purchase orders which signed by this agreement (hereinafter referred to as the "order"). 1.2 If the provisions of the agreement are inconsistent with the order, the order shall prevail. Not stated in order content will be subject to the provisions of agreement. Any modification, supplementary, give up should been written records, only to be valid by buyers and sellers authorized representative signature and confirmation, otherwise will be deemed invalid. 2. The agreement and order 2.1 During the validity term of this agreement, The buyer entrust SHENZHEN YICHANGTAI IMPORT AND EXPORT TRADE CO., LTD or SHENZHEN LEHEYUAN TRADING CO, LTD (her

In [2]:
test_dataset[1]

{'context': 'Exhibit 10.16 SUPPLY CONTRACT Contract No: Date: The buyer/End-User: Shenzhen LOHAS Supply Chain Management Co., Ltd. ADD: Tel No. : Fax No. : The seller: ADD: The Contract is concluded and signed by the Buyer and Seller on , in Hong Kong. 1. General provisions 1.1 This is a framework agreement, the terms and conditions are applied to all purchase orders which signed by this agreement (hereinafter referred to as the "order"). 1.2 If the provisions of the agreement are inconsistent with the order, the order shall prevail. Not stated in order content will be subject to the provisions of agreement. Any modification, supplementary, give up should been written records, only to be valid by buyers and sellers authorized representative signature and confirmation, otherwise will be deemed invalid. 2. The agreement and order 2.1 During the validity term of this agreement, The buyer entrust SHENZHEN YICHANGTAI IMPORT AND EXPORT TRADE CO., LTD or SHENZHEN LEHEYUAN TRADING CO, LTD (her

In [3]:
import os
os.environ["GOOGLE_CLOUD_PROJECT"] = os.environ["ANTHROPIC_VERTEX_PROJECT_ID"]
os.environ["GOOGLE_CLOUD_REGION"] = os.environ["CLOUD_ML_REGION"]

In [4]:
# Generate api completions
from anthropic import AsyncAnthropicVertex

client = AsyncAnthropicVertex(
    region=os.environ["GOOGLE_CLOUD_REGION"],
    project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
)

# simple completion
response = await client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=512,
    messages=[{'role': 'user', 'content': 'say: Hello, world!'}],
    system="You are a helpful assistant.",
)
response.content[0].text

'Hello, world!'

In [17]:
context = test_dataset[0]['context']
question = test_dataset[0]['question']
answer = test_dataset[0]['answer']

user_prompt_template = """# Context

{context}

---

Question: {question}

---

* Respond with the final answer inside the <final_answer> tag.
* Respond in bullet points.
* The answer should be as-is from the context.

Example:
<final_answer>
- answer 1
- answer 2
- answer 3
</final_answer>
"""

user_prompt = user_prompt_template.format(context=context, question=question)
user_prompt, user_prompt[-100:]


('# Context\n\nExhibit 10.16 SUPPLY CONTRACT Contract No: Date: The buyer/End-User: Shenzhen LOHAS Supply Chain Management Co., Ltd. ADD: Tel No. : Fax No. : The seller: ADD: The Contract is concluded and signed by the Buyer and Seller on , in Hong Kong. 1. General provisions 1.1 This is a framework agreement, the terms and conditions are applied to all purchase orders which signed by this agreement (hereinafter referred to as the "order"). 1.2 If the provisions of the agreement are inconsistent with the order, the order shall prevail. Not stated in order content will be subject to the provisions of agreement. Any modification, supplementary, give up should been written records, only to be valid by buyers and sellers authorized representative signature and confirmation, otherwise will be deemed invalid. 2. The agreement and order 2.1 During the validity term of this agreement, The buyer entrust SHENZHEN YICHANGTAI IMPORT AND EXPORT TRADE CO., LTD or SHENZHEN LEHEYUAN TRADING CO, LTD (h

In [12]:
# reasoning completion

response = await client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=11000,
    messages=[{'role': 'user', 'content': user_prompt}],
    thinking={"type": "enabled", "budget_tokens": 10000}, 
)
response

Message(id='msg_vrtx_01XFfCFzNNUKrWTbCKLL28su', container=None, content=[ThinkingBlock(signature='EukKCmUIDBACGAIqQJonHuvGrsPlL+jDyAqSOVUtNMRvfPXbzKK/ywvf4KMoEDSNuyd7S35Knk2hda/j0GayzjWM6LaxIIlxVjAoF2kyGWNsYXVkZS1oYWlrdS00LTUtMjAyNTEwMDE4ABIMdBmF/voSKpnqEJw2Ggxj8cCRRNANk+jX8vwiMIE1BX/nHCylX5YEmFyFLFKv5cKDtKVi1snMnMeHHK4Iyfvc/I3ihTo3tSXyxjN1ESqxCfM6vKBscYET7ZOrTWrfksCfMYW/9JyaEI+NFhioxMpSGyMd+AvIGAKbnwnDwuycra/PMoyh9LenOC/ZFGB4LFian9PbKlfxMByO90c87thN5jJmfRCyfF7Nxlu2EfzOWztU+4txwuFL8MMgmp+KnQHdimYLb2A2VDO/FlSVps2DgWRkYroUMOFYjcdRAE5MFExC5h6nC+MpIBM1Liy1wL0ZYr49mxfjmm96SXPm6otR8oLZt+bYp7BJ0e/NtnNVBrqzB7pCsfXUBFnh3agiEOO2wMJkfeykmbqlpPoF/qEmiXrXs1Qf+8w7nxuzECmGT4jUgt+6EbVzwPJF+mfgY4lDM0Uxyt5RGX6DGg+sxMI9J7Ure67IWnMW7rHpWNsaZ2gjjd7Cf/K42p8HekdN1rt8ap+5sQnR8zA6V7s5hh6oZXx/JRmqMTY4n7mNRmm+k7HxFWKqxLMOwOvqd1A0JJKqVCIeR4/3zR00LZOcDUhD9uyrY1JE/EFuQ+Klto/NFSf7OrYrBg/6DX1FoYJ51UqeZ0YvSlVsMQYVvFqpWAmPXlN/Ctd+AWpEn0EseBMhXAyPXfiFuhHpoSrIlGkHn2olxNaR7YwdC0s9C+jIJGipSu/zYCiQ7SrFElukzsIBfGOFmq6BLnPg5/

In [13]:
print(response.content[-1].text)
print('---')
print(answer)


<final_answer>
- Exhibit 10.16 SUPPLY CONTRACT (document title)
- Section 23: "This Contract is made out in three originals in both Chinese and English, each language being legally of the equal effect. Conflicts between these two languages arising there from, if any, shall be subject to Chinese version."
</final_answer>
---
['SUPPLY CONTRACT']


In [14]:

import re
pattern = re.compile(r'<final_answer>(.*?)</final_answer>', re.DOTALL)

def extract_answer(completion):
    matches = pattern.findall(completion)
    answer = matches[-1]
    answer = answer.strip().split('\n')
    answer = [a.strip() for a in answer if a.strip()]
    answer = [a[2:] for a in answer]
    return answer

extract_answer(response.content[-1].text)

['Exhibit 10.16 SUPPLY CONTRACT (document title)',
 'Section 23: "This Contract is made out in three originals in both Chinese and English, each language being legally of the equal effect. Conflicts between these two languages arising there from, if any, shall be subject to Chinese version."']

In [23]:
# evaluate

def evaluate_answer(prompts, completions, answers, **kwargs):
    """
    Evaluate the answer against the reference.
    """
    rewards = []
    for c, a in zip(completions, answers):
        try:
            c_answer = extract_answer(c[0]['content'])
            a_idx_to_c_idx = {}
            c_indices = set()
            for part in a:
                for c_idx, c_part in enumerate(c_answer):
                    if part in c_part and c_idx not in c_indices:
                        a_idx_to_c_idx[part] = c_idx
                        c_indices.add(c_idx)

            # if all parts are matched, and no extra parts are predicted
            if len(a_idx_to_c_idx) == len(a) and len(c_indices) == len(c_answer): rewards.append(1.0)
            else: rewards.append(0.0)
        except:
            rewards.append(0.0)

    return rewards

print(evaluate_answer([user_prompt], [[{'content': response.content[-1].text}]], [answer]))
print(evaluate_answer([user_prompt], [[{'content': '<final_answer>\n- SUPPLY CONTRACT\n</final_answer>'}]], [answer]))
print(evaluate_answer([user_prompt], [[{'content': '- SUPPLY CONTRACT'}]], [answer]))

[0.0]
[1.0]
[0.0]


In [19]:
test_dataset

Dataset({
    features: ['context', 'question', 'answer'],
    num_rows: 4182
})

In [22]:
test_dataset = test_dataset.map(lambda x: {'claude_messages': [{'role': 'user', 'content': user_prompt_template.format(context=x['context'], question=x['question'])}]})
test_dataset[0]

{'context': 'Exhibit 10.16 SUPPLY CONTRACT Contract No: Date: The buyer/End-User: Shenzhen LOHAS Supply Chain Management Co., Ltd. ADD: Tel No. : Fax No. : The seller: ADD: The Contract is concluded and signed by the Buyer and Seller on , in Hong Kong. 1. General provisions 1.1 This is a framework agreement, the terms and conditions are applied to all purchase orders which signed by this agreement (hereinafter referred to as the "order"). 1.2 If the provisions of the agreement are inconsistent with the order, the order shall prevail. Not stated in order content will be subject to the provisions of agreement. Any modification, supplementary, give up should been written records, only to be valid by buyers and sellers authorized representative signature and confirmation, otherwise will be deemed invalid. 2. The agreement and order 2.1 During the validity term of this agreement, The buyer entrust SHENZHEN YICHANGTAI IMPORT AND EXPORT TRADE CO., LTD or SHENZHEN LEHEYUAN TRADING CO, LTD (her

In [27]:
print(test_dataset[0]['question'])
test_dataset = test_dataset.shuffle(seed=42)
print(test_dataset[0]['question'])
# generate completions

Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
Highlight the parts (if any) of this contract related to "Cap On Liability" that should be reviewed by a lawyer. Details: Does the contract include a cap on liability upon the breach of a party’s obligation? This includes time limitation for the counterparty to bring claims or maximum amount for recovery.


In [31]:
test_dataset.select(range(50))

Dataset({
    features: ['context', 'question', 'answer', 'claude_messages'],
    num_rows: 50
})

In [32]:
# generate completions
import asyncio
semaphore = asyncio.Semaphore(40)
test_subset = test_dataset.select(range(50))

async def generate_completions(x):
    async with semaphore:
        try:
            response = await client.messages.create(
                model="claude-haiku-4-5",
                max_tokens=11000,
                messages=x['claude_messages'],
                thinking={"type": "enabled", "budget_tokens": 10000}, 
            )
            return response.content[-1].text
        except Exception as e:
            return f"Error generating completion for {x['question']}: {e}"

completions = await asyncio.gather(*[generate_completions(x) for x in test_subset])
completions[0]


'<final_answer>\n\n**Provisions Related to Cap on Liability:**\n\n- **Section 10.4(e) - Remedies for Monsanto**: "In the case of termination by Monsanto upon any of the Events of Default specified in Sections 10.4(b) (6), (7) and (9), the remedies of Monsanto shall be limited to (i) termination of this Agreement and (ii) the recovery of reasonable and customary out-of-pocket expenses incurred by Monsanto in transferring the Agent\'s duties hereunder to a new agent; provided that in no case shall the amount of expenses recoverable under this provision exceed $20MM."\n\n- **Section 10.4(f) - Exclusive Remedy**: "The payment of a Termination Fee to the Agent under Section 10.4(c) shall be deemed to constitute the exclusive remedy for any damages resulting out of the termination of this Agreement by Monsanto or the successor to the Roundup Business pursuant to Section 10.4(c) and the Agent shall waive its right to exercise any other remedies otherwise available at law or in equity."\n\n- *